In [ ]:
%%writefile ex1.cu

#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define N 10000000  // 10 Million

__global__ void cuda_func(char* arr) {
    int gid = threadIdx.x + blockIdx.x * blockDim.x;

    if (gid < N) {
        char val = arr[gid];

        if (val == 'A') {
            arr[gid] = 'T';
        } else if (val == 'C') {
            arr[gid] = 'G';
        } else if (val == 'G') {
            arr[gid] = 'C';
        } else {
            arr[gid] = 'A';
        }
    }
}

int main() {
    srand(time(NULL));

    char* h_arr = (char*)malloc(N * sizeof(char));

    for (int i = 0; i < N; i++) {
        int num = rand() % 4;

        if (num == 0) {
            h_arr[i] = 'A';
        } else if (num == 1) {
            h_arr[i] = 'C';
        } else if (num == 2) {
            h_arr[i] = 'G';
        } else {
            h_arr[i] = 'T';
        }
    }

    //First print
    printf("The initial DNA sequence \n");
    for (int i = 0; i < 50; i++) {
        printf("%c, ", h_arr[i]);
    }
    printf("\n");
    printf("------------------------- \n");

    // Initial arrays in the kernel

    char* d_arr_64; cudaMalloc((void**)&d_arr_64, N * sizeof(char));
    cudaMemcpy(d_arr_64, h_arr, N * sizeof(char), cudaMemcpyHostToDevice);

    char* d_arr_128; cudaMalloc((void**)&d_arr_128, N * sizeof(char));
    cudaMemcpy(d_arr_128, h_arr, N * sizeof(char), cudaMemcpyHostToDevice);

    char* d_arr_256; cudaMalloc((void**)&d_arr_256, N * sizeof(char));
    cudaMemcpy(d_arr_256, h_arr, N * sizeof(char), cudaMemcpyHostToDevice);

    char* d_arr_512; cudaMalloc((void**)&d_arr_512, N * sizeof(char));
    cudaMemcpy(d_arr_512, h_arr, N * sizeof(char), cudaMemcpyHostToDevice);

    //Thread sizes, block sizes, and result arrays

    int threadNum_64 = 64;
    int blockNum_64 = (N + threadNum_64 - 1) / threadNum_64;
    char* result_64 = (char*)malloc(N * sizeof(char));

    int threadNum_128 = 128;
    int blockNum_128 = (N + threadNum_128 - 1) / threadNum_128;
    char* result_128 = (char*)malloc(N * sizeof(char));

    int threadNum_256 = 256;
    int blockNum_256 = (N + threadNum_256 - 1) / threadNum_256;
    char* result_256 = (char*)malloc(N * sizeof(char));

    int threadNum_512 = 512;
    int blockNum_512 = (N + threadNum_512 - 1) / threadNum_512;
    char* result_512 = (char*)malloc(N * sizeof(char));

    // Execution and time measuring

    clock_t start_64 = clock();
    cuda_func<<<blockNum_64, threadNum_64>>>(d_arr_64);
    cudaDeviceSynchronize();
    clock_t end_64 = clock();

    clock_t start_128 = clock();
    cuda_func<<<blockNum_128, threadNum_128>>>(d_arr_128);
    cudaDeviceSynchronize();
    clock_t end_128 = clock();

    clock_t start_256 = clock();
    cuda_func<<<blockNum_256, threadNum_256>>>(d_arr_256);
    cudaDeviceSynchronize();
    clock_t end_256 = clock();

    clock_t start_512 = clock();
    cuda_func<<<blockNum_512, threadNum_512>>>(d_arr_512);
    cudaDeviceSynchronize();
    clock_t end_512 = clock();

    // Copying the results from the kernel

    cudaMemcpy(result_64, d_arr_64, N * sizeof(char), cudaMemcpyDeviceToHost);
    cudaMemcpy(result_128, d_arr_128, N * sizeof(char), cudaMemcpyDeviceToHost);
    cudaMemcpy(result_256, d_arr_256, N * sizeof(char), cudaMemcpyDeviceToHost);
    cudaMemcpy(result_512, d_arr_512, N * sizeof(char), cudaMemcpyDeviceToHost);

    //Results for each execution

    // 1
    printf("Results for execution with 64 threads in a block \n");
    for (int i = 0; i < 50; i++) {
        printf("%c, ", result_64[i]);
    }
    printf("\n");

    printf("Number of threads executed: %d \n", blockNum_64 * threadNum_64);
    printf("Execution time %f ms \n", (double)(end_64 - start_64));
    printf("------------------------- \n");

    // 2
    printf("Results for execution with 128 threads in a block \n");
    for (int i = 0; i < 50; i++) {
        printf("%c, ", result_128[i]);
    }
    printf("\n");

    printf("Number of threads executed: %d \n", blockNum_128 * threadNum_128);
    printf("Execution time %f ms \n", (double)(end_128 - start_128));
    printf("------------------------- \n");

    // 3
    printf("Results for execution with 256 threads in a block \n");
    for (int i = 0; i < 50; i++) {
        printf("%c, ", result_256[i]);
    }
    printf("\n");

    printf("Number of threads executed: %d \n", blockNum_256 * threadNum_256);
    printf("Execution time %f ms \n", (double)(end_256 - start_256));
    printf("------------------------- \n");

    // 4
    printf("Results for execution with 512 threads in a block \n");
    for (int i = 0; i < 50; i++) {
        printf("%c, ", result_512[i]);
    }
    printf("\n");

    printf("Number of threads executed: %d \n", blockNum_512 * threadNum_512);
    printf("Execution time %f ms \n", (double)(end_512 - start_512));

    free(h_arr);

    cudaFree(d_arr_64);
    cudaFree(d_arr_128);
    cudaFree(d_arr_256);
    cudaFree(d_arr_512);

    free(result_64);
    free(result_128);
    free(result_256);
    free(result_512);

}


Overwriting ex1.cu


In [ ]:
!nvcc ex1.cu -o Ex1

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./Ex1

The initial DNA sequence 
G, A, T, A, A, T, T, T, T, T, C, T, G, C, G, C, A, T, C, A, T, T, A, T, T, T, A, G, A, G, G, G, G, C, T, G, A, G, C, T, C, T, G, T, A, C, A, C, A, C, 
------------------------- 
Results for execution with 64 threads in a block 
C, T, A, T, T, A, A, A, A, A, G, A, C, G, C, G, T, A, G, T, A, A, T, A, A, A, T, C, T, C, C, C, C, G, A, C, T, C, G, A, G, A, C, A, T, G, T, G, T, G, 
Number of threads executed: 10000000 
Execution time 421.000000 ms 
------------------------- 
Results for execution with 128 threads in a block 
C, T, A, T, T, A, A, A, A, A, G, A, C, G, C, G, T, A, G, T, A, A, T, A, A, A, T, C, T, C, C, C, C, G, A, C, T, C, G, A, G, A, C, A, T, G, T, G, T, G, 
Number of threads executed: 10000000 
Execution time 236.000000 ms 
------------------------- 
Results for execution with 256 threads in a block 
C, T, A, T, T, A, A, A, A, A, G, A, C, G, C, G, T, A, G, T, A, A, T, A, A, A, T, C, T, C, C, C, C, G, A, C, T, C, G, A, G, A, C, A, T, G, T, G, T, G, 
N

In [ ]:
%%writefile ex2.cu

#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define N 20000000  // 20 Million

__global__ void cuda_func(char* arr, int* counter) {
    int gid = threadIdx.x + blockIdx.x * blockDim.x;

    if (gid < N) {
        char val = arr[gid];

        if (val == 'A') {
            atomicAdd(&counter[0], 1);
        } else if (val == 'C') {
            atomicAdd(&counter[1], 1);
        } else if (val == 'G') {
            atomicAdd(&counter[2], 1);
        } else {
            atomicAdd(&counter[3], 1);
        }
    }
}

int main() {
    srand(time(NULL));

    char* h_arr = (char*)malloc(N * sizeof(char));

    for (int i = 0; i < N; i++) {
        int num = rand() % 4;

        if (num == 0) {
            h_arr[i] = 'A';
        } else if (num == 1) {
            h_arr[i] = 'C';
        } else if (num == 2) {
            h_arr[i] = 'G';
        } else {
            h_arr[i] = 'T';
        }
    }

    // Counter {A, C, G, T} (an array is more convenient than 4 variables)
    int* h_counter = (int*)malloc(4 * sizeof(int));
    h_counter[0] = 0; h_counter[1] = 0; h_counter[2] = 0; h_counter[3] = 0;

    // Initial arrays in the kernel

    char* d_arr_64; cudaMalloc((void**)&d_arr_64, N * sizeof(char));
    cudaMemcpy(d_arr_64, h_arr, N * sizeof(char), cudaMemcpyHostToDevice);
    int* d_counter_64; cudaMalloc((void**)&d_counter_64, 4 * sizeof(int));
    cudaMemcpy(d_counter_64, h_counter, 4 * sizeof(int), cudaMemcpyHostToDevice);

    char* d_arr_128; cudaMalloc((void**)&d_arr_128, N * sizeof(char));
    cudaMemcpy(d_arr_128, h_arr, N * sizeof(char), cudaMemcpyHostToDevice);
    int* d_counter_128; cudaMalloc((void**)&d_counter_128, 4 * sizeof(int));
    cudaMemcpy(d_counter_128, h_counter, 4 * sizeof(int), cudaMemcpyHostToDevice);

    char* d_arr_256; cudaMalloc((void**)&d_arr_256, N * sizeof(char));
    cudaMemcpy(d_arr_256, h_arr, N * sizeof(char), cudaMemcpyHostToDevice);
    int* d_counter_256; cudaMalloc((void**)&d_counter_256, 4 * sizeof(int));
    cudaMemcpy(d_counter_256, h_counter, 4 * sizeof(int), cudaMemcpyHostToDevice);

    char* d_arr_512; cudaMalloc((void**)&d_arr_512, N * sizeof(char));
    cudaMemcpy(d_arr_512, h_arr, N * sizeof(char), cudaMemcpyHostToDevice);
    int* d_counter_512; cudaMalloc((void**)&d_counter_512, 4 * sizeof(int));
    cudaMemcpy(d_counter_512, h_counter, 4 * sizeof(int), cudaMemcpyHostToDevice);

    //Thread sizes, block sizes, and result arrays

    int threadNum_64 = 64;
    int blockNum_64 = (N + threadNum_64 - 1) / threadNum_64;

    int threadNum_128 = 128;
    int blockNum_128 = (N + threadNum_128 - 1) / threadNum_128;

    int threadNum_256 = 256;
    int blockNum_256 = (N + threadNum_256 - 1) / threadNum_256;

    int threadNum_512 = 512;
    int blockNum_512 = (N + threadNum_512 - 1) / threadNum_512;

    // Execution and time measuring

    clock_t start_64 = clock();
    cuda_func<<<blockNum_64, threadNum_64>>>(d_arr_64, d_counter_64);
    cudaDeviceSynchronize();
    clock_t end_64 = clock();

    clock_t start_128 = clock();
    cuda_func<<<blockNum_128, threadNum_128>>>(d_arr_128, d_counter_128);
    cudaDeviceSynchronize();
    clock_t end_128 = clock();

    clock_t start_256 = clock();
    cuda_func<<<blockNum_256, threadNum_256>>>(d_arr_256, d_counter_256);
    cudaDeviceSynchronize();
    clock_t end_256 = clock();

    clock_t start_512 = clock();
    cuda_func<<<blockNum_512, threadNum_512>>>(d_arr_512, d_counter_512);
    cudaDeviceSynchronize();
    clock_t end_512 = clock();

    //Copying the results back to the host

    int* h_counter_64 = (int*)malloc(4 * sizeof(int));
    cudaMemcpy(h_counter_64, d_counter_64, 4 * sizeof(int), cudaMemcpyDeviceToHost);

    int* h_counter_128 = (int*)malloc(4 * sizeof(int));
    cudaMemcpy(h_counter_128, d_counter_128, 4 * sizeof(int), cudaMemcpyDeviceToHost);

    int* h_counter_256 = (int*)malloc(4 * sizeof(int));
    cudaMemcpy(h_counter_256, d_counter_256, 4 * sizeof(int), cudaMemcpyDeviceToHost);

    int* h_counter_512 = (int*)malloc(4 * sizeof(int));
    cudaMemcpy(h_counter_512, d_counter_512, 4 * sizeof(int), cudaMemcpyDeviceToHost);

    //Results for each execution

    // 1
    printf("Results for execution with 64 threads in a block \n");
    printf("A: %d, C: %d, G: %d, T: %d \n", h_counter_64[0], h_counter_64[1], h_counter_64[2], h_counter_64[3]);
    printf("In total: %d \n", h_counter_64[0] + h_counter_64[1] + h_counter_64[2] + h_counter_64[3]);

    printf("Number of blocks : %d, Number of threads in each block: %d \n", blockNum_64, threadNum_64);
    printf("Execution time %f ms \n", (double)(end_64 - start_64));
    printf("------------------------- \n");

    // 2
    printf("Results for execution with 128 threads in a block \n");
    printf("A: %d, C: %d, G: %d, T: %d \n", h_counter_128[0], h_counter_128[1], h_counter_128[2], h_counter_128[3]);
    printf("In total: %d \n", h_counter_128[0] + h_counter_128[1] + h_counter_128[2] + h_counter_128[3]);

    printf("Number of blocks : %d, Number of threads in each block: %d \n", blockNum_128, threadNum_128);
    printf("Execution time %f ms \n", (double)(end_128 - start_128));
    printf("------------------------- \n");

    // 3
    printf("Results for execution with 256 threads in a block \n");
    printf("A: %d, C: %d, G: %d, T: %d \n", h_counter_256[0], h_counter_256[1], h_counter_256[2], h_counter_256[3]);
    printf("In total: %d \n", h_counter_256[0] + h_counter_256[1] + h_counter_256[2] + h_counter_256[3]);

    printf("Number of blocks : %d, Number of threads in each block: %d \n", blockNum_256, threadNum_256);
    printf("Execution time %f ms \n", (double)(end_256 - start_256));
    printf("------------------------- \n");

    // 4
    printf("Results for execution with 512 threads in a block \n");
    printf("A: %d, C: %d, G: %d, T: %d \n", h_counter_512[0], h_counter_512[1], h_counter_512[2], h_counter_512[3]);
    printf("In total: %d \n", h_counter_512[0] + h_counter_512[1] + h_counter_512[2] + h_counter_512[3]);

    printf("Number of blocks : %d, Number of threads in each block: %d \n", blockNum_512, threadNum_512);
    printf("Execution time %f ms \n", (double)(end_512 - start_512));
    printf("------------------------- \n");

    free(h_arr);
    free(h_counter);

    cudaFree(d_arr_64);
    cudaFree(d_arr_128);
    cudaFree(d_arr_256);
    cudaFree(d_arr_512);

    cudaFree(d_counter_64);
    cudaFree(d_counter_128);
    cudaFree(d_counter_256);
    cudaFree(d_counter_512);

    free(h_counter_64);
    free(h_counter_128);
    free(h_counter_256);
    free(h_counter_512);

}


Overwriting ex2.cu


In [ ]:
!nvcc ex2.cu -o Ex2

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./Ex2

Results for execution with 64 threads in a block 
A: 4998523, C: 4999097, G: 5000738, T: 5001642 
In total: 20000000 
Number of blocks : 312500, Number of threads in each block: 64 
Execution time 3927.000000 ms 
------------------------- 
Results for execution with 128 threads in a block 
A: 4998523, C: 4999097, G: 5000738, T: 5001642 
In total: 20000000 
Number of blocks : 156250, Number of threads in each block: 128 
Execution time 3690.000000 ms 
------------------------- 
Results for execution with 256 threads in a block 
A: 4998523, C: 4999097, G: 5000738, T: 5001642 
In total: 20000000 
Number of blocks : 78125, Number of threads in each block: 256 
Execution time 3681.000000 ms 
------------------------- 
Results for execution with 512 threads in a block 
A: 4998523, C: 4999097, G: 5000738, T: 5001642 
In total: 20000000 
Number of blocks : 39063, Number of threads in each block: 512 
Execution time 3681.000000 ms 
------------------------- 


In [ ]:
%%writefile ex3.cu
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define N 24000000  // 24 Million

__global__ void cuda_func(char* arr) {
    int gid = threadIdx.x + blockIdx.x * blockDim.x;

    if (gid < N) {
        int codon_id = gid / 3;
        int warp_id = threadIdx.x / 3;
        int thread_id = threadIdx.x % 3;
    }
}

int main() {
    srand(time(NULL));

    char* h_arr = (char*)malloc(N * sizeof(char));

    for (int i = 0; i < N; i++) {
        int num = rand() % 4;

        if (num == 0) {
            h_arr[i] = 'A';
        } else if (num == 1) {
            h_arr[i] = 'C';
        } else if (num == 2) {
            h_arr[i] = 'G';
        } else {
            h_arr[i] = 'T';
        }
    }

    char* d_arr; cudaMalloc((void**)&d_arr, N * sizeof(char));
    cudaMemcpy(d_arr, h_arr, N * sizeof(char), cudaMemcpyHostToDevice);

    int threadNum = 256;
    int blockNum = (N + threadNum - 1) / threadNum;

    cuda_func<<<blockNum, threadNum>>>(d_arr);
    cudaDeviceSynchronize();

    printf("Total number of codons: %d \n", N / 3);
    printf("Total number of threads: %d \n", blockNum * threadNum);
    printf("Total number of codons: %d \n", (threadNum / 32) * blockNum);

    free(h_arr);
    cudaFree(d_arr);

}

Overwriting ex3.cu


In [ ]:
!nvcc ex3.cu -o Ex3

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
ex3.cu(11): warning #177-D: variable "codon_id" was declared but never referenced
          int codon_id = gid / 3;
              ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

ex3.cu(12): warning #177-D: variable "warp_id" was declared but never referenced
          int warp_id = threadIdx.x / 3;
              ^

ex3.cu(13): warning #177-D: variable "thread_id" was declared but never referenced
          int thread_id = threadIdx.x % 3;
              ^



In [ ]:
!./Ex3

Total number of codons: 8000000 
Total number of threads: 24000000 
Total number of codons: 750000 
